# Dr Plant — Lemon exploratory baseline (Hugging Face source)

**This notebook deliberately does NOT use Mendeley dataset `8d9fv6kpt3` V3.** Its API requires OAuth2 and the anonymous request returns HTTP 401. Instead, this notebook uses the publicly hosted Hugging Face raw subset `Project-AgML/lemon_leaf_disease_classification` (upstream Mendeley DOI `10.17632/44nrn4593f.1`): 1,354 raw images across 9 classes, with CC BY 4.0 stated on the dataset card.

Only the raw subset is loaded; augmented images are excluded. This is a different, smaller dataset—not a replacement copy of the 17,586-image V3 collection. The resulting split is exploratory and is not independent validation for Syrian orchards.

Run from Google Colab on Android. The notebook downloads the raw subset directly from Hugging Face, converts it to class folders, caches the source and converted images in Google Drive, then audits and trains.

In [ ]:
# Select Runtime > Change runtime type > GPU (T4 if available) before running.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK = Path('/content/drplant-work')
DRIVE = Path('/content/drive/MyDrive/DrPlant')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)
print('GPU:', __import__('torch').cuda.is_available())

In [ ]:
# Clone the project reliably, including when this cell is re-run.
import os, subprocess
from pathlib import Path

REPO = Path('/content/drplant-source')
if (REPO / '.git').exists():
    print('Repository already exists; updating it...')
    update = subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], text=True, capture_output=True)
    if update.returncode != 0:
        raise RuntimeError('Could not update the existing Dr Plant checkout:\n' + update.stderr)
else:
    if REPO.exists():
        raise RuntimeError(f'{REPO} exists but is not a Git repository. In Colab, delete that folder from the Files panel or restart the runtime, then rerun this cell.')
    print('Downloading Dr Plant from GitHub...')
    clone = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jonbander/dr-plant.git', str(REPO)], text=True, capture_output=True)
    if clone.returncode != 0:
        raise RuntimeError('GitHub download failed. Check that Colab internet access is available, then rerun this cell. Details:\n' + clone.stderr)

os.chdir(REPO)
required = [Path('scripts/audit_image_classification_dataset.py'), Path('scripts/prepare_exploratory_splits.py'), Path('scripts/train_image_classifier.py')]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('The downloaded repository is missing required training scripts: ' + ', '.join(missing))

print('Project folder:', Path.cwd())
print('Required training scripts: OK')
subprocess.run([__import__('sys').executable, '-m', 'pip', 'install', '-q', 'Pillow', 'scikit-learn', 'datasets'], check=True)

In [ ]:
# Download the RAW subset from Hugging Face (no Mendeley API, login, ZIP upload, or manual Drive file).
# The source Parquet is cached in Drive, so a later Colab session can reuse it.
import json, re, shutil
from pathlib import Path
from PIL import Image
from datasets import load_dataset

HF_DATASET = 'Project-AgML/lemon_leaf_disease_classification'
DATASET_ID = '44nrn4593f'
VERSION = '1'
IMAGE_ROOT = DRIVE / 'lemon-agml-raw-images'
READY = IMAGE_ROOT / '.drplant-dataset-ready.json'
HF_CACHE = DRIVE / 'huggingface-cache'
HF_CACHE.mkdir(parents=True, exist_ok=True)

def existing_image_count(root):
    if not root.is_dir(): return 0
    return sum(1 for p in root.glob('*/*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})

if READY.is_file() and existing_image_count(IMAGE_ROOT) >= 100:
    saved = json.loads(READY.read_text(encoding='utf-8'))
    print('Reusing dataset already saved in Google Drive:', saved)
else:
    if IMAGE_ROOT.exists(): shutil.rmtree(IMAGE_ROOT)
    print('Connecting to public Hugging Face dataset:', HF_DATASET)
    ds = load_dataset(HF_DATASET, 'raw', split='train', cache_dir=str(HF_CACHE))
    label_feature = ds.features.get('label')
    if not hasattr(label_feature, 'names') or not label_feature.names:
        raise RuntimeError('The public dataset schema changed: expected a ClassLabel feature named label. Stop rather than guessing class names.')
    class_names = label_feature.names
    print('Raw images:', len(ds), '| classes:', len(class_names), '|', class_names)
    if len(ds) < 1000:
        raise RuntimeError(f'Unexpectedly small raw subset ({len(ds)} rows). Stop before training and inspect the Hugging Face dataset version.')
    IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    counts = {name: 0 for name in class_names}
    from io import BytesIO
    for i, row in enumerate(ds):
        label_index = int(row['label'])
        if label_index < 0 or label_index >= len(class_names):
            raise RuntimeError(f'Invalid class index at dataset row {i}: {label_index}')
        class_name = re.sub(r'[^A-Za-z0-9_-]+', '_', class_names[label_index]).strip('_').lower()
        image = row['image']
        if isinstance(image, dict):
            if image.get('bytes'):
                image = Image.open(BytesIO(image['bytes']))
            elif image.get('path'):
                image = Image.open(image['path'])
            else:
                raise RuntimeError(f'Image bytes/path missing at dataset row {i}')
        if not isinstance(image, Image.Image):
            raise RuntimeError(f'Unexpected image object at dataset row {i}: {type(image)}')
        target = IMAGE_ROOT / class_name
        target.mkdir(parents=True, exist_ok=True)
        image.convert('RGB').save(target / f'{counts[class_names[label_index]]:06d}.jpg', quality=95)
        counts[class_names[label_index]] += 1
        if (i + 1) % 200 == 0 or i + 1 == len(ds):
            print(f'Converted {i + 1}/{len(ds)} images')
    if sum(counts.values()) != len(ds) or len([n for n in counts.values() if n]) != len(class_names):
        raise RuntimeError('Converted image count or class coverage does not match the source dataset.')
    saved = {'dataset': HF_DATASET, 'upstream_doi': '10.17632/44nrn4593f.1', 'subset': 'raw', 'rows_converted': len(ds), 'class_counts': counts}
    READY.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Dataset converted and saved to Drive:', IMAGE_ROOT)

print('Image files available:', existing_image_count(IMAGE_ROOT))

In [ ]:
# Verify the saved class-folder layout before continuing.
image_ext = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
classes = sorted(p for p in IMAGE_ROOT.iterdir() if p.is_dir())
counts = {p.name: sum(1 for f in p.rglob('*') if f.is_file() and f.suffix.lower() in image_ext) for p in classes}
counts = {k:v for k,v in counts.items() if v}
if len(counts) != 9 or sum(counts.values()) < 1000:
    raise RuntimeError(f'Dataset folder validation failed. Found {sum(counts.values())} images in {len(counts)} classes; expected the raw 9-class dataset. Details: {counts}')
print('Validated image root:', IMAGE_ROOT)
print('Image count:', sum(counts.values()))
print('Class counts:', counts)

In [ ]:
# Inventory, SHA-256 duplicate scan, and image decoding check.
AUDIT = WORK / 'audit'
!python scripts/audit_image_classification_dataset.py "$IMAGE_ROOT" --dataset-id 44nrn4593f --version 1 --license 'CC BY 4.0 (Hugging Face card; upstream Mendeley DOI 10.17632/44nrn4593f.1)' --out "$AUDIT" --verify-images
import json
audit_report = json.loads((AUDIT/'archive-audit-report.json').read_text())
print('Images:', audit_report['image_count'])
print('Class counts:', audit_report['class_image_counts'])
print('Exact duplicate groups:', audit_report['exact_duplicate_groups'])
print('Unreadable images:', len(audit_report['unreadable_images']))
if audit_report['image_count'] < 1000: raise RuntimeError('Unexpectedly small raw dataset; stop and inspect before training.')

In [ ]:
# Exploratory split: exact byte-identical files stay together; near-duplicates and site/plant/session leakage are NOT ruled out.
SPLITS = WORK / 'exploratory-splits'
!python scripts/prepare_exploratory_splits.py "$AUDIT/image-manifest.csv" --out "$SPLITS" --seed 42
print((SPLITS/'exploratory-split-report.json').read_text())

In [ ]:
# Train a first MobileNetV3-Small baseline. Test metrics are exploratory, not independent validation.
OUTPUT = WORK / 'model-output'
!python scripts/train_image_classifier.py "$IMAGE_ROOT" "$SPLITS/exploratory-splits.csv" --out "$OUTPUT" --epochs 12 --batch-size 32 --pretrained --exploratory-non-independent
print((OUTPUT/'metrics.json').read_text())

In [ ]:
# Preserve outputs in Drive without deleting earlier runs.
import shutil
DEST = DRIVE / 'lemon-agml-raw-exploratory-baseline'
DEST.mkdir(parents=True, exist_ok=True)
for item in OUTPUT.iterdir():
    target = DEST / item.name
    if item.is_dir():
        if target.exists(): shutil.rmtree(target)
        shutil.copytree(item, target)
    else:
        shutil.copy2(item, target)
for source in (AUDIT/'archive-audit-report.json', SPLITS/'exploratory-split-report.json'):
    shutil.copy2(source, DEST/source.name)
shutil.copy2(READY, DEST/READY.name)
print('Saved to Google Drive:', DEST)
print('This is an exploratory baseline from the AgML raw subset—not the Mendeley V3 dataset and not independent validation on Syrian farms.')